# Requests & BeautifulSoup

Making HTTP requests and parsing HTML to extract web data.

**Install:** `pip install requests beautifulsoup4 lxml`

**In this notebook:**
- GET requests and response object
- Query parameters and headers
- Session and error handling
- BeautifulSoup — finding elements
- CSS selectors
- Extracting structured data
- Scraping pattern

## 1. Basic GET Request

In [ ]:
import requests

r = requests.get('https://httpbin.org/get', timeout=10)

print('Status:', r.status_code)   # 200
print('OK:', r.ok)                 # True
print('URL:', r.url)
print('Content-Type:', r.headers['Content-Type'])

# JSON body
data = r.json()
print('Origin IP:', data['origin'])

## 2. Query Parameters and Headers

In [ ]:
import requests

# Query params — requests builds the URL for you
r = requests.get(
    'https://httpbin.org/get',
    params={'language': 'Python', 'level': 3},
    headers={'User-Agent': 'MyBot/1.0'},
    timeout=10
)
print('URL:', r.url)            # includes ?language=Python&level=3
print('Args:', r.json()['args'])
print('Headers sent:', r.json()['headers']['User-Agent'])

## 3. POST Request and Error Handling

In [ ]:
import requests
from requests.exceptions import HTTPError, Timeout, ConnectionError

# POST with JSON body
r = requests.post('https://httpbin.org/post',
                  json={'name': 'Alice', 'score': 99},
                  timeout=10)
print(r.json()['json'])   # echoes back the body

# raise_for_status — auto-raise on 4xx/5xx
try:
    r = requests.get('https://httpbin.org/status/404', timeout=10)
    r.raise_for_status()
except HTTPError as e:
    print(f'HTTP {e.response.status_code}: {e}')
except Timeout:
    print('Timed out')

## 4. Session — Reuse Connection and Headers

In [ ]:
import requests

with requests.Session() as s:
    s.headers.update({'User-Agent': 'PythonScraper/1.0'})

    r1 = s.get('https://httpbin.org/headers', timeout=10)
    r2 = s.get('https://httpbin.org/get', timeout=10)

    # Both responses show the shared User-Agent header
    print(r1.json()['headers']['User-Agent'])
    print(r2.json()['headers']['User-Agent'])

## 5. BeautifulSoup — Parsing HTML

In [ ]:
from bs4 import BeautifulSoup

html = """
<html>
<head><title>My Page</title></head>
<body>
  <nav><a href='/home'>Home</a> <a href='/about'>About</a></nav>
  <article class='post'>
    <h2>Python Tips</h2>
    <p class='intro'>Python is great.</p>
    <p>Use virtual environments.</p>
  </article>
</body>
</html>
"""

soup = BeautifulSoup(html, 'lxml')

# Finding elements
print(soup.title.text)                         # My Page
print(soup.find('h2').text)                    # Python Tips
print([a['href'] for a in soup.find_all('a')]) # ['/home', '/about']

# CSS selectors
print(soup.select_one('article.post h2').text)
print([p.get_text(strip=True) for p in soup.select('article p')])

## 6. Extracting Structured Data

In [ ]:
from bs4 import BeautifulSoup

table_html = """
<table>
  <thead><tr><th>Name</th><th>Score</th></tr></thead>
  <tbody>
    <tr><td>Alice</td><td>92</td></tr>
    <tr><td>Bob</td><td>87</td></tr>
  </tbody>
</table>
"""

soup  = BeautifulSoup(table_html, 'lxml')
hdrs  = [th.text.lower() for th in soup.select('thead th')]
rows  = []
for tr in soup.select('tbody tr'):
    cells = [td.text for td in tr.select('td')]
    rows.append(dict(zip(hdrs, cells)))

print(rows)
# [{'name': 'Alice', 'score': '92'}, {'name': 'Bob', 'score': '87'}]

## 7. Live Scraping Pattern

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def scrape_page(url):
    r = requests.get(url, timeout=10, headers={'User-Agent': 'PythonScraper/1.0'})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'lxml')
    books = []
    for article in soup.select('article.product_pod'):
        books.append({
            'title':  article.h3.a['title'],
            'price':  article.select_one('.price_color').text.strip(),
            'rating': article.p['class'][1],
        })
    return books

# Scrape page 1 of books.toscrape.com
books = scrape_page('https://books.toscrape.com/catalogue/page-1.html')
print(f'Found {len(books)} books')
for b in books[:3]:
    print(b)

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | GET requests, status codes, JSON APIs |
| [02-medium.py](exercises/02-medium.py) | Medium | Session, error handling, HTML parsing |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | Multi-page scraper, pagination, CSV export |

Solutions: [solutions/](solutions/)